# 04 — Seleção dos participantes e cálculo da conectividade funcional
1. Localiza `sfnwmrda<ID>_session_1_rest_1_cc200_TCs.1D` (filtrado, sessão 1, repouso 1)
2. Cruza com o fenotípico
3. Mantém `QC_Athena == 1` e `DX ∈ {0,1,2,3}`
4. Define `classe` (0 = controle, 1 = TDAH)
5. Calcula e salva a matriz de correlação (190×190) de cada participante

In [ ]:
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from nilearn.connectome import ConnectivityMeasure
import config

config.criar_pastas()
pheno = pd.read_csv(config.ARQ_PHENO, sep="\t")
pheno["DX"] = pd.to_numeric(pheno["DX"], errors="coerce")   # 'pending' vira NaN
pheno["ID_padronizado"] = pheno["ScanDir ID"].astype("Int64").astype(str)

In [ ]:
registros = []
for p in config.EXTRAIDO.rglob("sfnwmrda*_session_1_rest_1_cc200_TCs.1D"):
    registros.append({"site": p.parents[1].name, "subject_id": p.parent.name,
                      "arquivo": p.name, "caminho": str(p)})
df_exames = pd.DataFrame(registros)
df_exames["ID_padronizado"] = pd.to_numeric(df_exames["subject_id"], errors="coerce").astype("Int64").astype(str)
print("Arquivos CC200 filtrados (session 1 / rest 1):", len(df_exames))
print("IDs únicos:", df_exames["ID_padronizado"].nunique())

In [ ]:
df_part = df_exames.merge(pheno, on="ID_padronizado", how="left", suffixes=("_exame", "_pheno"))
print("Exames:", len(df_exames), "| após merge:", len(df_part))
print("Sem DX:", df_part["DX"].isna().sum(), "| sem QC_Athena:", df_part["QC_Athena"].isna().sum())
print("\nDX:\n", df_part["DX"].value_counts(dropna=False))
display(pd.crosstab(df_part["DX"], df_part["QC_Athena"], margins=True))

In [ ]:
df_sel = df_part[(df_part["QC_Athena"] == 1) & (df_part["DX"].isin([0, 1, 2, 3]))].copy()
df_sel["classe"] = (df_sel["DX"] != 0).astype(int)

print("Total selecionados:", len(df_sel))
print("\nDiagnóstico original:\n", df_sel["DX"].value_counts().sort_index())
print("\nClasses finais:\n", df_sel["classe"].value_counts().sort_index())

In [ ]:
def calcular_conectividade(caminho):
    df = pd.read_csv(caminho, sep="\t")
    df.columns = [str(c).strip() for c in df.columns]
    cols = [c for c in df.columns if c.startswith("Mean_")]
    ts = df[cols].apply(pd.to_numeric, errors="coerce").values

    if ts.shape[1] != config.N_ROIS:
        raise ValueError(f"Número inesperado de ROIs: {ts.shape[1]}")
    if not np.isfinite(ts).all():
        raise ValueError("Série temporal contém NaN/Inf.")

    fc = ConnectivityMeasure(kind="correlation").fit_transform([ts])[0]
    if not np.allclose(fc, fc.T):
        raise ValueError("Matriz FC não simétrica.")
    if not np.isfinite(fc).all():
        raise ValueError("Matriz FC contém NaN/Inf.")

    roi_ids = np.array([int(c.split("_")[1]) for c in cols])
    return fc, roi_ids

In [ ]:
erros, processados, roi_ref = [], 0, None

for _, r in tqdm(df_sel.iterrows(), total=len(df_sel), desc="Calculando conectividade"):
    try:
        fc, roi_ids = calcular_conectividade(r["caminho"])
        if roi_ref is None:
            roi_ref = roi_ids
        elif not np.array_equal(roi_ref, roi_ids):
            raise ValueError("Conjunto de ROIs difere do primeiro participante.")
        np.save(config.CONECT / f"FC_{r['ID_padronizado']}.npy", fc)
        processados += 1
    except Exception as e:
        erros.append({"ID": r["ID_padronizado"], "arquivo": r["arquivo"], "erro": str(e)})

print("Processados:", processados, "| Erros:", len(erros))
if erros:
    display(pd.DataFrame(erros))

In [ ]:
ids_ok = {p.stem.replace("FC_", "") for p in config.CONECT.glob("FC_*.npy")}
df_sel = df_sel[df_sel["ID_padronizado"].isin(ids_ok)].reset_index(drop=True)

df_sel.to_csv(config.ARQ_PARTICIPANTES, index=False)
np.save(config.ARQ_ROIS, roi_ref)
print("Salvo:", config.ARQ_PARTICIPANTES, "|", len(df_sel), "participantes")
print("Salvo:", config.ARQ_ROIS, "|", len(roi_ref), "ROIs")